# **1: Create Masks from Fluorescence Z-Stacks**

### **Goal:**
For each fluorescence Z-stack, produce a binary mask that segments the vacuoles. This mask will be used for training the U-Net model later on.

### **Pipeline:**
1. Load fluorescence Z-stack
2. Background subtraction (Gaussian blur)
3. Maximum intensity projection
4. Thresholding (binary mask)
5. Clean up mask with morphological operations
6. Save mask as TIFF

## **0. Imports**

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from skimage import io, filters, morphology, exposure
from skimage.filters import threshold_otsu, threshold_li
import tifffile

## **1. Config**

In [4]:
# Input paths
FLUO_PATHS = [
    Path("images/nn60ap6log1_w2SDC-488_s1.stk"),
    Path("images/nn60ap6log1_w2SDC-488_s2.stk"),
    Path("images/nn60ap6log1_w2SDC-488_s3.stk"),
    Path("images/nn60ap6log1_w2SDC-488_s4.stk"),
    Path("images/nn60ap6log1_w2SDC-488_s5.stk"),
]

# Output paths
MASK_DIR = Path("masks")
MASK_DIR.mkdir(exist_ok=True)

# Parameters for pipeline. Parameters were determined by looking at the images in Fiji
GAUSSIAN_SIGMA = 50 # for background subtraction
MIN_OBJECT_SIZE = 5 # for cleanup

## **2. Image Processing functions**

In [5]:
def subtract_background_max_z_projection(stack, sigma):
    """
    Background subtraction using a Gaussian blur. We accumulate directly into the MIP to save RAM.
    Output: z projection (np.ndarray) of the stack with background removed
    """
    mip = np.zeros(stack.shape[1:], dtype=np.float32)  # (Y, X) only

    for z in range(stack.shape[0]):
        slice_ = stack[z].astype(np.float32)
        background = filters.gaussian(slice_, sigma=sigma)
        corrected = np.clip(slice_ - background, 0, None)
        mip = np.maximum(mip, corrected)

    return mip

def threshold_image(image):
    """
    Converts a grayscale image to a binary mask by thresholding.
    Uses an otsu thresholding method
    """
    thresh = threshold_otsu(image)
    mask = image > thresh
    return mask, thresh


def clean_mask(mask, min_size):
    """
    Remove small objects from the binary mask. 
    (Removes anything below min_size pixels as it is likely noise)
    """
    return morphology.remove_small_objects(mask, min_size=min_size)

## **3. Process images**

In [ ]:
results = []  # store the max intensity projection and the mask for visualisation

for idx, fluo_path in enumerate(FLUO_PATHS):
    print(f"Processing image {idx+1}/{len(FLUO_PATHS)}: {fluo_path.name}")

    # Load the fluorescence Z-stack
    stack = tifffile.imread(fluo_path)

    # Background subtract + z projection
    mip = subtract_background_max_z_projection(stack, sigma=GAUSSIAN_SIGMA)

    # delete the raw stack to free memory
    del stack

    # Thresholding
    mask, thresh = threshold_image(mip)

    # Cleanup to remove small particles / noise
    mask_clean = clean_mask(mask, min_size=MIN_OBJECT_SIZE)

    # Save the mask
    mask_name = fluo_path.stem + "_mask.tif"
    mask_path = MASK_DIR / mask_name
    tifffile.imwrite(mask_path, mask_clean.astype(np.uint8) * 255)

    # Store the results
    results.append({"name": fluo_path.stem, "mip": mip, "mask": mask_clean})
    print(f"  Done. Threshold={thresh:.2f}, "
          f"Foreground={mask_clean.mean()*100:.1f}%")

Processing image 1/5: nn60ap6log1_w2SDC-488_s1.stk


## **4. Visualizations**

In [ ]:
# Display z projection and mask side by side for every image
fig, axes = plt.subplots(len(results), 3, figsize=(14, 4 * len(results)))

for row, r in enumerate(results):
    mip = r["mip"]
    mask = r["mask"]

    # Column 0: z projection
    axes[row, 0].imshow(exposure.rescale_intensity(mip), cmap="bone")
    axes[row, 0].set_title(f"{r['name']}\nMax Z-projection")
    axes[row, 0].axis("off")

    # Column 1: binary mask
    axes[row, 1].imshow(mask, cmap="gray")
    axes[row, 1].set_title("Binary mask")
    axes[row, 1].axis("off")

    # Column 2: overlay — MIP in green, mask contour in red
    axes[row, 2].imshow(exposure.rescale_intensity(mip), cmap="bone")
    axes[row, 2].contour(mask, colors="cyan", linewidths=0.8)
    axes[row, 2].set_title("Overlay (mask contour)")
    axes[row, 2].axis("off")

plt.tight_layout()
plt.savefig(MASK_DIR / "mask_QC.png", dpi=150)
plt.show()

## **5. Summary**

Masks are saved in `masks/` as `<image_name>_mask.tif`.

**Next step:** 
Open `02_unet_training.ipynb` to train the U-Net using the brightfield images and the masks from this notebook.